# 07. Probability Calibration Analysis: Heart Disease Screening

**Phase**: Heart Disease — Phase 7  
**Document Reference**: `docs/HEART_CALIBRATION_ANALYSIS.md`  
**Parent Artifacts**: `artifacts/heart/experiments/`, `artifacts/heart/threshold/`  
**New Phase 7 Artifacts**: `artifacts/heart/calibration/`  
**Dataset**: Cleveland Heart Disease (`datasets/raw/heart/processed.cleveland.data`)  
**Strict Phase Boundary**: Empirical probability calibration only. No SHAP explainability (Phase 8), no model retraining of candidate spaces, no production deployment or FastAPI bundling.

---

### Section 1: Objective & Calibration Background
In clinical binary screening, classification models produce a continuous output $p \in [0, 1]$. While Phase 5 optimized **discrimination** (the model's capacity to rank individuals with coronary artery disease above individuals without disease, measured by ROC-AUC and PR-AUC), Phase 7 evaluates **empirical calibration** (the degree to which model-estimated probability values correspond to observed outcome frequencies).

**Critical Distinction**:
- **Discrimination**: If patient A has higher disease risk than patient B, is $\hat{p}_A > \hat{p}_B$?
- **Calibration**: Among patients assigned an estimated probability near $0.80$, do approximately $80\%$ actually have coronary artery disease?

> [!IMPORTANT]
> **Clinical Non-Claim**:
> Model-estimated probabilities reflect empirical frequencies on the Cleveland Heart Disease cohort under specific feature distributions. They do **not** represent true biological individual patient risk, and calibrated probabilities do not constitute clinical diagnosis or proof of disease presence/absence.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add ml-service to Python path
sys.path.append(os.path.abspath("../ml-service"))

from app.heart_preprocessing import (
    verify_raw_dataset_integrity,
    load_heart_raw_data,
    create_modeling_dataset,
    split_heart_data,
)
from app.heart_calibration import (
    get_heart_finalist_configs,
    compute_ece,
    calculate_calibration_metrics,
    run_training_calibration_cv,
    fit_calibrated_models_on_train,
    evaluate_calibrated_models_on_holdout,
    analyze_threshold_shifts,
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

print("Libraries imported successfully. Reproducible random seed:", RANDOM_STATE)


### Section 2: Dataset Integrity & Leakage-Free Train/Holdout Separation

To guarantee zero data leakage:
1. Raw dataset cryptographic hash (`processed.cleveland.data`) is verified against its reference SHA-256 and MD5.
2. The dataset is split into an 80% training cohort ($N=242$) and a 20% holdout test set ($N=61$) stratified by the binary target.
3. The 61-sample holdout is strictly locked and **never seen** during calibration method selection, cross-fitting, or hyperparameter optimization.


In [ ]:
# 1. Verify Raw Dataset
is_valid, md5_hash, sha_hash, file_sz = verify_raw_dataset_integrity(
    os.path.abspath("../datasets/raw/heart/processed.cleveland.data")
)
print(f"Dataset Cryptographic Verification: Valid={is_valid}")
print(f"  MD5:    {md5_hash}")
print(f"  SHA256: {sha_hash}")
print(f"  Size:   {file_sz} bytes")
assert is_valid, "Cryptographic integrity failed!"

# 2. Load and Split
df_raw = load_heart_raw_data(os.path.abspath("../datasets/raw/heart/processed.cleveland.data"))
X, y = create_modeling_dataset(df_raw)
X_train, X_test, y_train, y_test = split_heart_data(X, y, test_size=0.20, random_state=RANDOM_STATE)

print(f"\nCohort Split:")
print(f"  Training cohort: {len(X_train)} samples ({y_train.sum()} positive, {(y_train == 0).sum()} negative)")
print(f"  Hold-out cohort: {len(X_test)} samples ({y_test.sum()} positive, {(y_test == 0).sum()} negative)")
assert len(X_train) == 242 and len(X_test) == 61


### Section 3: Finalist Architectures & Calibration Methodology

We evaluate the three finalist model configurations established in Phase 5:
1. **Logistic Regression (Robust L1)**: `penalty='l1', C=1.0, solver='liblinear'` with RobustScaler.
2. **Random Forest (n=200, leaf=5)**: `n_estimators=200, min_samples_leaf=5` with tree preprocessor.
3. **XGBoost (n=100, depth=2, lr=0.05)**: `n_estimators=100, max_depth=2, learning_rate=0.05` with tree preprocessor.

#### Calibration Approaches Compared:
- **A. Uncalibrated Baseline**: Raw predicted probabilities from the model pipeline.
- **B. Sigmoid / Platt Scaling**: Parametric logistic sigmoid transform:
  $$\hat{P}(Y=1 \mid s) = \frac{1}{1 + \exp(A \cdot s + B)}$$
  Fitted via maximum likelihood; stable with small sample sizes ($N=242$).
- **C. Isotonic Regression**: Non-parametric piecewise constant monotonic step function:
  $$\min_{\hat{m}_1 \le \dots \le \hat{m}_n} \sum_{i=1}^n (y_i - \hat{m}_i)^2$$
  Highly flexible, but carries severe risk of step-boundary overfitting in small cohorts ($N=242$).


In [ ]:
candidate_configs = get_heart_finalist_configs()

# Run 5-fold Stratified Cross-Validation on the 242-sample training cohort
print("Executing 5-fold Stratified CV on training cohort (N=242) for calibration development...")
train_summary_df, oof_probs_df = run_training_calibration_cv(
    X_train, y_train, candidate_configs, n_splits=5, random_state=RANDOM_STATE
)

cols_display = [
    "model", "calibration_method", "brier_score", "log_loss",
    "ece", "cal_intercept", "cal_slope", "roc_auc", "pr_auc"
]
print("\n--- TRAINING COHORT (CV OUT-OF-FOLD) CALIBRATION METRICS ---")
display_train = train_summary_df[cols_display].copy()
display_train


### Section 4: Training Cohort (CV) Empirical Findings & Isotonic Overfitting

#### Key Observations on Development Cohort ($N=242$):
1. **Logistic Regression**:
   - The uncalibrated baseline is already exceptionally well-calibrated: $\text{Brier} = 0.1188$, $\text{Log Loss} = 0.3848$, $\text{ECE} = 0.0383$, and Calibration Slope $= 1.0704$ (ideal is $1.00$).
   - Sigmoid calibration slightly widens the slope to $1.3396$ with minor metric trade-offs.
   - **Isotonic calibration exhibits step overfitting**: While training Brier score is $0.1153$, the cross-validated Log Loss spikes to $0.4985$ (+29.5%) due to step-boundary overconfidence on out-of-fold evaluations.
2. **Random Forest**:
   - Uncalibrated tree ensembles exhibit characteristic probability compression towards the base rate ($0.50$), reflected in a calibration slope of $1.5774$ and $\text{ECE} = 0.0900$.
   - **Sigmoid calibration significantly improves Random Forest**: Brier score drops from $0.1328$ to $0.1285$, Log Loss drops from $0.4208$ to $0.4053$, ECE improves from $0.0900$ to $0.0634$, and the calibration slope adjusts to $1.1996$.
   - Isotonic calibration again severely inflates cross-validated Log Loss to $0.6543$ (+55.5%), demonstrating clear non-parametric overfitting.
3. **XGBoost**:
   - The uncalibrated shallow gradient boosted trees ($d=2, \eta=0.05$) are well-calibrated natively: $\text{Brier} = 0.1313$, $\text{Log Loss} = 0.4131$, $\text{ECE} = 0.0659$, and Slope $= 1.0877$.
   - Sigmoid calibration maintains near-identical performance (ECE $0.0643$).
   - Isotonic calibration elevates Log Loss to $0.5466$.


In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)
methods = ["Uncalibrated", "Sigmoid", "Isotonic"]
colors = {"Uncalibrated": "#7f8c8d", "Sigmoid": "#2980b9", "Isotonic": "#27ae60"}
markers = {"Uncalibrated": "o", "Sigmoid": "s", "Isotonic": "^"}

for idx, cfg in enumerate(candidate_configs):
    ax = axes[idx]
    c_name = cfg["candidate_name"]
    s_name = cfg["short_name"]
    
    ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration (Ideal)", alpha=0.7)
    
    for method in methods:
        probs = oof_probs_df[f"{c_name}__{method}"].values
        prob_true, prob_pred = calibration_curve(y_train.values, probs, n_bins=10, strategy="uniform")
        
        row = train_summary_df[(train_summary_df["model"] == s_name) & (train_summary_df["calibration_method"] == method)].iloc[0]
        label = f"{method} (Brier={row['brier_score']:.3f}, ECE={row['ece']:.3f})"
        ax.plot(prob_pred, prob_true, marker=markers[method], color=colors[method], linewidth=2, label=label)
        
    ax.set_title(f"{s_name}\n(Training Cohort CV, N=242)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Mean Predicted Probability", fontsize=11)
    if idx == 0:
        ax.set_ylabel("Observed Positive Frequency", fontsize=11)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.legend(loc="upper left", fontsize=9, frameon=True)
    ax.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Heart Disease Phase 7 — Reliability Diagrams (Development CV, N=242)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


### Section 5: Descriptive Hold-Out Evaluation ($N=61$)

Following methodological freeze on the training cohort:
- Final calibrated pipelines are refitted on the full 242 training instances.
- The untouched 61-sample holdout ($33$ controls, $28$ positive cases) is evaluated **strictly once and descriptively**.
- Holdout metrics are reported to assess generalization across Brier score, Log Loss, and discrimination.
- **Rule**: Holdout results are purely descriptive and are not used to choose the calibration algorithm.


In [ ]:
# Fit final pipelines on full 242-sample training set
fitted_models = fit_calibrated_models_on_train(X_train, y_train, candidate_configs, cv=5)

# Evaluate strictly on locked holdout
holdout_summary_df, test_preds_df = evaluate_calibrated_models_on_holdout(
    fitted_models, X_test, y_test, candidate_configs
)

print("\n--- HOLDOUT COHORT (61 SAMPLES) CALIBRATION METRICS ---")
display_holdout = holdout_summary_df[cols_display].copy()
display_holdout


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), sharey=True)

for idx, cfg in enumerate(candidate_configs):
    ax = axes[idx]
    c_name = cfg["candidate_name"]
    s_name = cfg["short_name"]
    
    ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration (Ideal)", alpha=0.7)
    
    for method in methods:
        probs = test_preds_df[f"{c_name}__{method}"].values
        prob_true, prob_pred = calibration_curve(y_test.values, probs, n_bins=5, strategy="uniform")
        
        row = holdout_summary_df[(holdout_summary_df["model"] == s_name) & (holdout_summary_df["calibration_method"] == method)].iloc[0]
        label = f"{method} (Brier={row['brier_score']:.3f}, LL={row['log_loss']:.3f})"
        ax.plot(prob_pred, prob_true, marker=markers[method], color=colors[method], linewidth=2, label=label)
        
    ax.set_title(f"{s_name}\n(Locked Hold-Out, N=61)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Mean Predicted Probability", fontsize=11)
    if idx == 0:
        ax.set_ylabel("Observed Positive Frequency", fontsize=11)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.legend(loc="upper left", fontsize=9, frameon=True)
    ax.grid(True, linestyle="--", alpha=0.5)

plt.suptitle("Heart Disease Phase 7 — Descriptive Reliability Diagrams (Hold-Out, N=61)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 11), sharex=True, sharey=True)

for row_idx, cfg in enumerate(candidate_configs):
    c_name = cfg["candidate_name"]
    s_name = cfg["short_name"]
    
    for col_idx, method in enumerate(methods):
        ax = axes[row_idx, col_idx]
        probs = test_preds_df[f"{c_name}__{method}"].values
        
        sns.histplot(
            probs[y_test.values == 0],
            bins=np.linspace(0, 1, 11),
            color="#3498db",
            label="Control (y=0)",
            alpha=0.6,
            ax=ax,
            stat="count"
        )
        sns.histplot(
            probs[y_test.values == 1],
            bins=np.linspace(0, 1, 11),
            color="#e74c3c",
            label="Heart Disease (y=1)",
            alpha=0.6,
            ax=ax,
            stat="count"
        )
        
        if row_idx == 0:
            ax.set_title(f"{method} Calibration", fontsize=12, fontweight="bold")
        if col_idx == 0:
            ax.set_ylabel(f"{s_name}\nCount", fontsize=11, fontweight="bold")
        else:
            ax.set_ylabel("")
        if row_idx == 2:
            ax.set_xlabel("Predicted Probability", fontsize=11)
        ax.set_xlim([0, 1])
        ax.legend(loc="upper center", fontsize=9)
        ax.grid(True, linestyle=":", alpha=0.5)

plt.suptitle("Probability Distributions by True Class (Hold-Out, N=61)", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()


### Section 6: Relationship to Phase 6 Operating Thresholds

In Phase 6, decision cutoffs were optimized on raw uncalibrated probabilities:
- **Logistic Regression**: High-Sensitivity $t=0.31$, Balanced Youden $t=0.56$
- **Random Forest**: High-Sensitivity $t=0.32$, Balanced Youden $t=0.55$
- **XGBoost**: High-Sensitivity $t=0.24$, Balanced Youden $t=0.62$

**How Calibration Alters Decision Cutoffs**:
Probability calibration transforms the continuous numerical mapping from raw scores to estimated probabilities. While ranking order is largely preserved (ROC-AUC remains virtually identical), the numerical cutoff associated with a specific operational goal (such as $\ge 90\%$ sensitivity) shifts:


In [ ]:
shift_path = os.path.join("../ml-service/artifacts/heart/calibration/threshold_shift_analysis.csv")
shift_df = pd.read_csv(shift_path)

shift_display_cols = [
    "model", "calibration_method",
    "phase6_uncal_high_sens_threshold", "calibrated_high_sens_threshold",
    "calibrated_high_sens_recall", "calibrated_high_sens_specificity",
    "phase6_uncal_youden_threshold", "calibrated_youden_threshold",
    "calibrated_youden_recall", "calibrated_youden_specificity"
]
print("--- OPERATING THRESHOLD SHIFTS POST-CALIBRATION ---")
shift_df[shift_display_cols]


### Section 7: Discussion & Limitations

1. **Cohort Sample Size ($N=242$ Train, $N=61$ Holdout)**:
   - In small cohorts, non-parametric calibration (Isotonic Regression) is prone to step-boundary overfitting. Although isotonic regression appears competitive on holdout Brier score, its cross-validated log-loss is substantially worse (+29% to +55% penalty).
   - Parametric Sigmoid (Platt) scaling is markedly more robust for small samples.
2. **Discrimination vs Calibration**:
   - Monotonic probability calibration does not change sample rank ordering; ROC-AUC is preserved.
   - However, calibrated probabilities provide more reliable risk stratum mapping for thresholding and downstream risk communication.
3. **Clinical Boundaries**:
   - Calibrated outputs represent empirical conditional frequencies on the Cleveland cohort, not patient-specific certainty.
   - Outputs should be framed as "model-estimated probabilities" rather than "true diagnostic risk."

---

### Section 8: Conclusion & Phase 8 Handoff
- **Empirical Calibration Selection**:
  - For **Logistic Regression**, uncalibrated probabilities are already well-calibrated (slope $\approx 1.07$, ECE $\approx 0.038$).
  - For **Random Forest**, **Sigmoid (Platt) calibration** substantially improves probability calibration, reducing uncalibrated overconfidence and adjusting the calibration slope from $1.58$ to $1.20$.
  - For **XGBoost**, uncalibrated probabilities and Sigmoid-calibrated probabilities demonstrate strong calibration consistency.
- **Handoff to Phase 8**:
  - The calibrated probability models will be integrated into the explainability analysis in **Phase 8 (SHAP Explainability & Feature Attributions)**.
